# 4단계 — 엣지별 경사도 결합

2단계에서 만든 도로망 그래프의 각 엣지(도로 한 조각)에 경사도(%)를 붙인다.

왜 이 방식(등고선+표고점 보간 -> 노드 높이 추정 -> 엣지 경사도 계산)을 쓰는지는 `step4_경사도_결합_설명.md`에 정리했다.

핵심 로직은 `compute_edge_slope.py`에 있고, 이 노트북은 그 모듈을 불러와 실행하고 결과를 확인하는 역할만 한다.

In [ ]:
import sys
from pathlib import Path

import osmnx as ox

sys.path.append(str(Path.cwd()))
import collect_infra_places as infra
import snap_infra_to_network as snap
import compute_edge_slope as slope

## 2단계 그래프 불러오기

In [ ]:
G = ox.load_graphml(snap.GRAPH_PATH)
G_proj = ox.project_graph(G)
print(f"그래프: 노드 {len(G.nodes):,}개, 엣지 {len(G.edges):,}개")

그래프: 노드 576개, 엣지 1,626개


## 높이 산점 데이터 확인

서울시 경사도 등고선(선 위의 각 정점)과 표고점(점)을 합쳐 (x, y, 높이) 산점 데이터를 만든다. bbox 경계 근처 노드의 보간이 안정적이도록 경계 바깥 300m까지 같이 가져온다 (설명 문서 3장).

In [ ]:
xs, ys, hs = slope.load_elevation_samples(infra.TARGET_BBOX, G_proj.graph["crs"])
print(f"높이 산점 데이터: {len(xs):,}개")
print(f"높이 범위: {hs.min():.1f}m ~ {hs.max():.1f}m")

높이 산점 데이터: 52,315개
높이 범위: 5.0m ~ 173.1m


## 노드별 높이 추정

그래프의 각 노드 위치에서, 주변 산점 데이터를 선형보간해 높이를 추정한다 (범위 밖이라 선형보간이 안 되는 노드는 최근접값으로 채움).

In [ ]:
node_elev = slope.estimate_node_elevations(G_proj, xs, ys, hs)

import numpy as np
elev_values = np.array(list(node_elev.values()))
print(f"노드 높이 추정 범위: {elev_values.min():.1f}m ~ {elev_values.max():.1f}m (평균 {elev_values.mean():.1f}m)")

노드 높이 추정 범위: 10.3m ~ 127.9m (평균 40.8m)


## 엣지 경사도 계산

엣지 양 끝 노드의 높이차 / 엣지 길이로 경사도(%)를 계산해 그래프에 속성으로 붙인다.

In [ ]:
G_scored = slope.add_slope_to_edges(G, node_elev)
ox.save_graphml(G_scored, slope.GRAPH_WITH_SLOPE_PATH)
print(f"저장: {slope.GRAPH_WITH_SLOPE_PATH}")

slopes = np.array([d["slope_pct"] for _, _, d in G_scored.edges(data=True)])
print(
    f"경사도(%) 요약 - 평균: {slopes.mean():.1f}, 중앙값: {np.median(slopes):.1f}, "
    f"최대: {slopes.max():.1f}, 10% 초과 엣지 수: {(slopes > 10).sum()} / {len(slopes)}"
)

저장: C:\Users\kjyil\Documents\python_study\DartB_TOYPROJECT\data\pedestrian_graph_with_slope.graphml
경사도(%) 요약 - 평균: 5.7, 중앙값: 3.7, 최대: 61.8, 10% 초과 엣지 수: 282 / 1626


## 지도로 확인

엣지를 경사도 구간별로 색칠해서 실제로 가파른 길이 어디인지 눈으로 확인한다. (초록 <5%, 노랑 5~10%, 주황 10~15%, 빨강 15%+)

In [ ]:
import folium

def slope_color(pct):
    if pct < 5:
        return "green"
    if pct < 10:
        return "gold"
    if pct < 15:
        return "orange"
    return "red"

m = folium.Map(location=[infra.CENTER_LAT, infra.CENTER_LON], zoom_start=15, tiles="cartodbpositron")

for u, v, data in G_scored.edges(data=True):
    if "geometry" in data:
        coords = [(lat, lon) for lon, lat in data["geometry"].coords]
    else:
        coords = [(G_scored.nodes[u]["y"], G_scored.nodes[u]["x"]), (G_scored.nodes[v]["y"], G_scored.nodes[v]["x"])]
    folium.PolyLine(
        coords,
        color=slope_color(data["slope_pct"]),
        weight=3,
        opacity=0.8,
        popup=f"경사도 {data['slope_pct']:.1f}%",
    ).add_to(m)

folium.Marker(
    [infra.CENTER_LAT, infra.CENTER_LON], popup="중앙대 정문", icon=folium.Icon(color="blue"),
).add_to(m)

m

C:\Users\kjyil\AppData\Local\Programs\Python\Python314\Lib\site-packages\folium\raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore
